# 06 - RAG Evaluation, Failure Modes, And Best Practices

## Learning objectives

- Evaluate retrieval relevance and answer faithfulness.
- Use OpenAI and Claude as evaluator models in mock-safe form.
- Identify common RAG failure modes.
- Design better tests for grounded AI systems.

## Concept primer

RAG systems can fail in two different places. Retrieval can fetch the wrong evidence. Generation can ignore good evidence or invent unsupported claims. Evaluation should inspect both stages.


## Step 1 - Setup


In [ ]:
# This setup cell keeps imports working whether Jupyter starts in the repo root
# or inside the nlp_transformers_embeddings folder.
from pathlib import Path
import os
import sys

CURRENT = Path.cwd()
COURSE_ROOT = CURRENT.parent if CURRENT.name == "nlp_transformers_embeddings" else CURRENT
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dotenv import load_dotenv
load_dotenv(COURSE_ROOT / ".env", override=False)
load_dotenv(COURSE_ROOT / "nlp_transformers_embeddings" / ".env", override=False)

# Security practice: print whether keys are present, never the key values.
print("LIVE_API enabled:", os.getenv("LIVE_API", "false").lower() == "true")
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("ANTHROPIC_API_KEY loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


## Step 2 - Create a deliberately imperfect answer

A good evaluation lesson includes a failure. Here the answer contains one supported claim and one unsupported claim.


In [ ]:
from nlp_transformers_embeddings.utils.mock_data import SAMPLE_DOCUMENTS
from nlp_transformers_embeddings.utils.retrieval import chunk_documents, rank_chunks
from nlp_transformers_embeddings.utils.openai_client import embed_texts, generate_json
from nlp_transformers_embeddings.utils.anthropic_client import critique_grounding
from nlp_transformers_embeddings.utils.schemas import EvaluationScore

question = "Does Anthropic provide first-party embeddings?"
chunks = chunk_documents(SAMPLE_DOCUMENTS, chunk_size_words=55, overlap_words=8)
chunk_embeddings = embed_texts([chunk.text for chunk in chunks])
retrieved = rank_chunks(embed_texts([question])[0], chunk_embeddings, chunks, top_k=3)

evidence = "\n\n".join(f"[{item.rank}] {item.chunk.text}" for item in retrieved)
answer = "Anthropic provides Claude for generation and also has a built-in first-party embeddings API for vectors."

print("Question:", question)
print("Answer under test:", answer)
print("\nEvidence:\n", evidence)


## Step 3 - Claude-style grounding critique

Claude is useful as a careful evaluator. In live mode, this calls the Anthropic API. In mock mode, it returns a deterministic critique structure.


In [ ]:
critique = critique_grounding(question=question, answer=answer, evidence=evidence)
print(critique)


## Step 4 - OpenAI structured evaluation

Structured output turns an evaluator response into a machine-readable score. This makes dashboards, regression tests, and grading rubrics easier.


In [ ]:
evaluation_schema = {
    "type": "object",
    "properties": {
        "evaluator": {"type": "string"},
        "faithfulness": {"type": "number"},
        "relevance": {"type": "number"},
        "citation_quality": {"type": "number"},
        "notes": {"type": "string"},
    },
    "required": ["evaluator", "faithfulness", "relevance", "citation_quality", "notes"],
    "additionalProperties": False,
}

mock_eval = {
    "evaluator": "openai-mock",
    "faithfulness": 0.35,
    "relevance": 0.75,
    "citation_quality": 0.2,
    "notes": "The answer is relevant but incorrectly claims Anthropic has first-party embeddings.",
}

payload = generate_json(
    prompt=f"""Evaluate this answer against the evidence.
Question: {question}
Answer: {answer}
Evidence: {evidence}""",
    schema_name="rag_evaluation",
    schema=evaluation_schema,
    mock_payload=mock_eval,
)
score = EvaluationScore.model_validate(payload)
print(score.model_dump())


## Step 5 - Failure-mode table

Students should leave this module with names for common failures. Naming the failure makes it easier to debug.


In [ ]:
failure_modes = [
    ("Bad retrieval", "The right evidence never enters the prompt", "Improve chunking, embeddings, filters, or query rewriting"),
    ("Context overload", "Too much weak evidence distracts the model", "Retrieve fewer, higher-quality passages"),
    ("Unsupported synthesis", "The model adds plausible facts not in evidence", "Require citations and run faithfulness evaluation"),
    ("Stale corpus", "The documents are outdated", "Track source timestamps and refresh indexes"),
    ("Secret leakage", "A key or private data appears in a notebook", "Use .env, masking, and repository secret scanning"),
]

for name, symptom, mitigation in failure_modes:
    print(f"{name}: {symptom}. Mitigation: {mitigation}.")


## Student exercise

Create your own bad answer for a retrieved passage. Score it manually before running the evaluator. Did your judgment match the model-based evaluation?

## Debugging checklist

- If eval scores are always high, add adversarial examples.
- If eval scores are always low, clarify the rubric.
- If evaluators disagree, inspect the evidence and define tie-break rules.

## Production best practices

- Keep a fixed evaluation set and run it after prompt or model changes.
- Evaluate retrieval and generation separately.
- Store evaluator model name, rubric version, and timestamp.
- Treat model-based evals as signals, not unquestionable truth.

## Reflection questions

1. What is the difference between relevance and faithfulness?
2. Why is a cited answer still not automatically correct?
3. What failure would be most damaging in a student-facing assistant?
